In [128]:
import pandas as pd
import numpy as np
import re

from nltk import word_tokenize
from sklearn.metrics import f1_score

import warnings
warnings.filterwarnings("ignore")

seed = 123

In [129]:
# load dataset
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

# rename target col to labels
train = train.rename(columns={"target": "labels"})
test["labels"] = 0
ytrain = train["labels"]

In [130]:
# clean text, store cleaned as col, "text_cleaned"
def clean_text(text_in:str):
    out = text_in
    out = re.sub("\n", " ", out) # strip multiline

    out = re.sub("B1G", "BIG", out) # strip multiline

    out = re.sub(r"http[^\s]+", " ", out) # strip http* link
    out = re.sub(r'--&gt;', '', out)
    out = out.replace(r'&lt;', '')
    out = out.replace(r'&gt;', '')
    out = out.replace(r'&amp;', '')
    out = re.sub(r"@\w+", "@USER", out) # @___ -> @USER
    out = re.sub(r"[" + re.escape('!"$%&()*+,-./:;<=>?[\\]^_`{|}~') + "]", " ", out) # most punctuation -> space
    out = re.sub(r"\d+", " ", out) # strip numbers
    out = re.sub(" ' ", " ", out) # most punctuation -> space

    # final space processing
    out = re.sub(r"\s+", " ", out) # multispace to 1 space
    out = re.sub(r"^\s+", "", out) # trim leading space
    out = re.sub(r"\s+$", "", out) # trim trailing space

    return out

train["text_cleaned"] = train["text"].apply(clean_text)
test["text_cleaned"] = test["text"].apply(clean_text)

# print before and after
_train_index = None
_test_index = None

# _train_index = 5803
# _test_index = 2125

_tmp_train = train.sample(1) if _train_index is None else train.loc[[_train_index]]
print("[Train] Row Index: ", _tmp_train.index[0])
print("[Train] Before clean: ", _tmp_train["text"].values[0])
print("[Train] After clean: ", _tmp_train["text_cleaned"].values[0])
print("===========================================================")
_tmp_test = test.sample(1) if _test_index is None else test.loc[[_test_index]]
print("[Test] Row Index: ", _tmp_test.index[0])
print("[Test] Before clean: ", _tmp_test["text"].values[0])
print("[Test] After clean: ", _tmp_test["text_cleaned"].values[0])

# clean tmp var
del _tmp_train, _tmp_test, _train_index, _test_index

[Train] Row Index:  1693
[Train] Before clean:  #TheDoolingGroup 2 injured when 2 school buses collide - åÊ #BREAKING: School bus slams into school bus in Bordento... http://t.co/YQHfio9XQm
[Train] After clean:  #TheDoolingGroup injured when school buses collide åÊ #BREAKING School bus slams into school bus in Bordento
[Test] Row Index:  1710
[Test] Before clean:  Standing in mountaintop field scented with wildflowers dust and distant forest fires smell of impending rain in the air.  #fieldworksmells
[Test] After clean:  Standing in mountaintop field scented with wildflowers dust and distant forest fires smell of impending rain in the air #fieldworksmells


In [131]:
# new feature
def count_pattern(pattern:str, text:str):
    return len(re.findall(pattern, text))

for df in [train, test]:
    df["mention_count"] = df["text"].apply(lambda x: count_pattern("@\w+", x))
    df["link_count"] = df["text"].apply(lambda x: count_pattern(r"http[^\s]+", x))
    df["exclamation_count"] = df["text"].apply(lambda x: count_pattern(r"!", x))
    df["char_count"] = df["text_cleaned"].apply(len)
    df["word_count"] = df["text_cleaned"].apply(lambda x: len(word_tokenize(x)))
    df["hashtag_count"] = df["text"].apply(lambda x: count_pattern(r"#\w+", x))
    df["uppercase_count"] = df["text"].apply(lambda x: len(re.findall(r"[A-Z]", x)))
    df["lowercase_count"] = df["text"].apply(lambda x: len(re.findall(r"[a-z]", x)))
    df["upper-to-all-ratio"] = df.apply(lambda x: x["uppercase_count"] / x["char_count"] if x["char_count"] > 0 else 0, axis=1)
    df["lower-to-all-ratio"] = df.apply(lambda x: x["lowercase_count"] / x["char_count"] if x["char_count"] > 0 else 0, axis=1)

In [132]:
# feature selection
selected_feature = [
    "mention_count",
    "link_count",
    "exclamation_count",
    "char_count",
    "word_count",
    "hashtag_count",
    "uppercase_count",
    "lowercase_count",
    "upper-to-all-ratio",
    "lower-to-all-ratio",
]

xtrain = train[selected_feature]
xtest = test[selected_feature]

In [133]:
# logistic analysis
from statsmodels import api as sm
logit_model = sm.Logit(ytrain, sm.add_constant(xtrain))
result = logit_model.fit()
result.summary2()

Optimization terminated successfully.
         Current function value: 0.629818
         Iterations 5


<class 'statsmodels.iolib.summary2.Summary'>
"""
                          Results: Logit
===================================================================
Model:               Logit            Method:           MLE        
Dependent Variable:  labels           Pseudo R-squared: 0.078      
Date:                2025-10-30 18:02 AIC:              9611.6104  
No. Observations:    7613             BIC:              9687.9241  
Df Model:            10               Log-Likelihood:   -4794.8    
Df Residuals:        7602             LL-Null:          -5201.3    
Converged:           1.0000           LLR p-value:      3.1816e-168
No. Iterations:      5.0000           Scale:            1.0000     
-------------------------------------------------------------------
                    Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
-------------------------------------------------------------------
const              -0.9001   0.2418  -3.7221 0.0002 -1.3740 -0.4261
mention_count       0.0615   0.0680   0.9049 0.3655 -0.0717  0.1947
link_count          0.5262   0.1498   3.5117 0.0004  0.2325  0.8199
exclamation_count  -0.2189   0.0489  -4.4723 0.0000 -0.3148 -0.1230
char_count          0.0452   0.0103   4.3994 0.0000  0.0250  0.0653
word_count         -0.2064   0.0156 -13.2561 0.0000 -0.2369 -0.1759
hashtag_count       0.0866   0.0237   3.6566 0.0003  0.0402  0.1330
uppercase_count     0.0099   0.0113   0.8760 0.3810 -0.0122  0.0319
lowercase_count    -0.0011   0.0102  -0.1061 0.9155 -0.0211  0.0189
upper-to-all-ratio -1.3386   0.4124  -3.2460 0.0012 -2.1469 -0.5303
lower-to-all-ratio -0.1475   0.2566  -0.5751 0.5652 -0.6504  0.3553
===================================================================

"""

In [134]:
# drop mention_count, uppercase_count, lowercase_count, lower-to-all-ratio	
drop_col = ["mention_count", "uppercase_count", "lowercase_count", "lower-to-all-ratio"]
xtrain_selected = xtrain.drop(columns=drop_col)
xtest_selected = xtest.drop(columns=drop_col)

logit_model = sm.Logit(ytrain, sm.add_constant(xtrain_selected))
result = logit_model.fit()
result.summary2()

Optimization terminated successfully.
         Current function value: 0.630335
         Iterations 5


<class 'statsmodels.iolib.summary2.Summary'>
"""
                          Results: Logit
===================================================================
Model:               Logit            Method:           MLE        
Dependent Variable:  labels           Pseudo R-squared: 0.077      
Date:                2025-10-30 18:02 AIC:              9611.4801  
No. Observations:    7613             BIC:              9660.0434  
Df Model:            6                Log-Likelihood:   -4798.7    
Df Residuals:        7606             LL-Null:          -5201.3    
Converged:           1.0000           LLR p-value:      1.1533e-170
No. Iterations:      5.0000           Scale:            1.0000     
-------------------------------------------------------------------
                    Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
-------------------------------------------------------------------
const              -1.1314   0.0858 -13.1842 0.0000 -1.2996 -0.9632
link_count          0.4805   0.0439  10.9432 0.0000  0.3945  0.5666
exclamation_count  -0.2143   0.0487  -4.3990 0.0000 -0.3097 -0.1188
char_count          0.0449   0.0023  19.7595 0.0000  0.0404  0.0493
word_count         -0.2009   0.0118 -17.0260 0.0000 -0.2240 -0.1778
hashtag_count       0.0902   0.0234   3.8522 0.0001  0.0443  0.1360
upper-to-all-ratio -0.5212   0.1878  -2.7752 0.0055 -0.8892 -0.1531
===================================================================

"""

In [135]:
# from catboost import CatBoostClassifier
# from sklearn.model_selection import StratifiedKFold
# kfold = StratifiedKFold(n_splits=5, random_state=seed)

# for train_idx, val_idx in kfold.split(xtrain_selected, ytrain):
#     X_tr, X_val = xtrain_selected.iloc[train_idx], xtrain_selected.iloc[val_idx]
#     y_tr, y_val = ytrain.iloc[train_idx], ytrain.iloc[val_idx]
    
#     model = CatBoostClassifier(
#         iterations=1000,
#         learning_rate=0.05,
#         depth=6,
#         eval_metric='AUC',
#         random_seed=seed,
#         early_stopping_rounds=50,
#         verbose=100
#     )
    
#     model.fit(
#         X_tr, y_tr,
#         eval_set=(X_val, y_val),
#         use_best_model=True
#     )

In [136]:
# LGBM performed the bese among models with logit output
from lazypredict.Supervised import LazyClassifier
from sklearn.model_selection import train_test_split

# split train to train and eval
_xtrain, _xval, _ytrain, _yval = train_test_split(
    xtrain_selected, ytrain, test_size=0.2, random_state=123, stratify=ytrain
)

clf = LazyClassifier(
    random_state=seed, verbose=0, ignore_warnings=True, custom_metric=None
)
models, predictions = clf.fit(_xtrain, _xval, _ytrain, _yval)
del _xtrain, _xval, _ytrain, _yval
models

  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 2617, number of negative: 3473
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000128 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 455
[LightGBM] [Info] Number of data points in the train set: 6090, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.429721 -> initscore=-0.282990
[LightGBM] [Info] Start training from score -0.282990


,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
LabelSpreading,0.70,0.69,0.69,0.69,0.93
LabelPropagation,0.69,0.69,0.69,0.69,0.75
LGBMClassifier,0.69,0.68,0.68,0.69,0.06
QuadraticDiscriminantAnalysis,0.68,0.68,0.68,0.68,0.01
SVC,0.69,0.67,0.67,0.68,1.38
KNeighborsClassifier,0.68,0.67,0.67,0.67,0.03
AdaBoostClassifier,0.68,0.67,0.67,0.67,0.17
GaussianNB,0.67,0.66,0.66,0.67,0.01
NearestCentroid,0.66,0.66,0.66,0.66,0.02


In [138]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold

FOLD = 5

kfold = StratifiedKFold(n_splits=FOLD)
oof_train_logit = np.zeros((len(xtrain_selected),)) # logit of labels=1
oof_test_logit = np.zeros((FOLD, len(xtest_selected))) # logit of labels=1

for fold, (train_idx, val_idx) in enumerate(kfold.split(xtrain_selected, ytrain)):
    print(f"=== FOLD {fold+1} ===")
    X_tr, X_val = xtrain_selected.iloc[train_idx], xtrain_selected.iloc[val_idx]
    y_tr, y_val = ytrain.iloc[train_idx], ytrain.iloc[val_idx]
    
    model = LGBMClassifier(
        learning_rate=0.05,
        random_state=seed,
    )
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='f1_score',
    )
    
    oof_train_logit[val_idx] = model.predict_proba(X_val)[:,1]
    oof_test_logit[fold, :] = model.predict_proba(xtest_selected)[:,1]


=== FOLD 1 ===
[LightGBM] [Info] Number of positive: 2616, number of negative: 3474
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000177 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 453
[LightGBM] [Info] Number of data points in the train set: 6090, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.429557 -> initscore=-0.283660
[LightGBM] [Info] Start training from score -0.283660
=== FOLD 2 ===
[LightGBM] [Info] Number of positive: 2617, number of negative: 3473
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000132 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 454
[LightGBM] [Info] Number of data points in the train set: 6090, number of used features: 

In [139]:
# save oof
np.save("saved_var/lgbm/oof_train_logit.npy", oof_train_logit)
np.save("saved_var/lgbm/oof_test_logit.npy", oof_test_logit)